In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import boto3
import os
import json
import time
import random
import plotly.express as px
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text, inspect

In [2]:
load_dotenv()

def upload_file_to_s3(local_path, folder):
    """Upload a local CSV file to the specified S3 folder."""
    bucket = os.getenv("S3_BUCKET")
    prefix = os.getenv(f"S3_KEY_{folder.upper()}")
    filename = os.path.basename(local_path)

    s3 = boto3.client("s3")
    s3.upload_file(
        Filename=local_path,
        Bucket=bucket,
        Key=f"{prefix}{filename}",
        ExtraArgs={"ContentType": "text/csv"}
    )

    print(f"Uploaded {filename} to s3://{bucket}/{prefix}{filename}")

### Latitude and longitude (GPS coordinates) of the 35 cities using the Nominatim API.

In [4]:
c = "Annecy"
url = 'https://nominatim.openstreetmap.org/search?'

params = {
    "q": c,
    "format": "json",
    "limit": 1
}

headers = {
    "User-Agent": "KayakProjectDataPipeline/1.0 (contact@example.com)"
}


r = requests.get(url, params=params, headers=headers)
print(r.status_code)
data = r.json()

200


In [5]:
data

[{'place_id': 80790948,
  'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
  'osm_type': 'relation',
  'osm_id': 6791758,
  'lat': '45.8992348',
  'lon': '6.1288847',
  'class': 'boundary',
  'type': 'administrative',
  'place_rank': 16,
  'importance': 0.6203828713594545,
  'addresstype': 'city',
  'name': 'Annecy',
  'display_name': 'Annecy, Haute-Savoie, Auvergne-Rhône-Alpes, France métropolitaine, France',
  'boundingbox': ['45.8280024', '45.9766928', '6.0484121', '6.2043932']}]

In [6]:
cities = [
    "Mont Saint Michel", "Saint Malo", "Bayeux",
    "Le Havre", "Rouen", "Paris", "Amiens",
    "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon",
    "Annecy", "Grenoble", "Lyon", "Gorges du Verdon",
    "Bormes les Mimosas", "Cassis", "Marseille",
    "Aix en Provence", "Avignon", "Uzes",
    "Nimes", "Aigues Mortes", "Saintes Maries de la mer",
    "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz",
    "Bayonne", "La Rochelle"
]

coords = []
headers = {
        "User-Agent": "KayakProjectDataPipeline/1.0 (contact@example.com)"
    }
url = 'https://nominatim.openstreetmap.org/search?'

for city in cities:
    
    params = {
        "q": city,
        "format": "json",
        "limit": 1,
        "addressdetails": 0
    }
    try:
        response = requests.get(url, params=params, 
                                headers=headers, timeout=10)
        print(response.status_code)
        data = response.json()
        if data:
            coords.append({
                "city": city,
                "lat": data[0]["lat"],
                "lon": data[0]["lon"]
            })
            print(f"{city} : lat= {data[0]['lat']} lon= {data[0]['lon']}")
        else:
            print(f"No data found for {city}")
    except Exception as e:
        print(f"Error for {city}: {e}")

    time.sleep(1.2)


200
Mont Saint Michel : lat= 48.6359541 lon= -1.5114600
200
Saint Malo : lat= 48.6495180 lon= -2.0260409
200
Bayeux : lat= 49.2764624 lon= -0.7024738
200
Le Havre : lat= 49.4938975 lon= 0.1079732
200
Rouen : lat= 49.4404591 lon= 1.0939658
200
Paris : lat= 48.8588897 lon= 2.3200410
200
Amiens : lat= 49.8941708 lon= 2.2956951
200
Lille : lat= 50.6365654 lon= 3.0635282
200
Strasbourg : lat= 48.5846140 lon= 7.7507127
200
Chateau du Haut Koenigsbourg : lat= 48.2494107 lon= 7.3443202
200
Colmar : lat= 48.0777517 lon= 7.3579641
200
Eguisheim : lat= 48.0447968 lon= 7.3079618
200
Besancon : lat= 47.2380222 lon= 6.0243622
200
Dijon : lat= 47.3215806 lon= 5.0414701
200
Annecy : lat= 45.8992348 lon= 6.1288847
200
Grenoble : lat= 45.1875602 lon= 5.7357819
200
Lyon : lat= 45.7578137 lon= 4.8320114
200
Gorges du Verdon : lat= 43.7496562 lon= 6.3285616
200
Bormes les Mimosas : lat= 43.1506968 lon= 6.3419285
200
Cassis : lat= 43.2140359 lon= 5.5396318
200
Marseille : lat= 43.2961743 lon= 5.3699525
200


In [7]:
len(coords)

35

In [8]:
coords

[{'city': 'Mont Saint Michel', 'lat': '48.6359541', 'lon': '-1.5114600'},
 {'city': 'Saint Malo', 'lat': '48.6495180', 'lon': '-2.0260409'},
 {'city': 'Bayeux', 'lat': '49.2764624', 'lon': '-0.7024738'},
 {'city': 'Le Havre', 'lat': '49.4938975', 'lon': '0.1079732'},
 {'city': 'Rouen', 'lat': '49.4404591', 'lon': '1.0939658'},
 {'city': 'Paris', 'lat': '48.8588897', 'lon': '2.3200410'},
 {'city': 'Amiens', 'lat': '49.8941708', 'lon': '2.2956951'},
 {'city': 'Lille', 'lat': '50.6365654', 'lon': '3.0635282'},
 {'city': 'Strasbourg', 'lat': '48.5846140', 'lon': '7.7507127'},
 {'city': 'Chateau du Haut Koenigsbourg',
  'lat': '48.2494107',
  'lon': '7.3443202'},
 {'city': 'Colmar', 'lat': '48.0777517', 'lon': '7.3579641'},
 {'city': 'Eguisheim', 'lat': '48.0447968', 'lon': '7.3079618'},
 {'city': 'Besancon', 'lat': '47.2380222', 'lon': '6.0243622'},
 {'city': 'Dijon', 'lat': '47.3215806', 'lon': '5.0414701'},
 {'city': 'Annecy', 'lat': '45.8992348', 'lon': '6.1288847'},
 {'city': 'Grenoble

In [9]:
df_coords = pd.DataFrame(coords)

In [10]:
df_coords.head()

,city,lat,lon
0,Mont Saint Michel,48.6359541,-1.5114600
1,Saint Malo,48.6495180,-2.0260409
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658


In [11]:
df_coords.columns

Index(['city', 'lat', 'lon'], dtype='object')

#### Create an id for each city

In [12]:
df_coords["city_id"] = [
    f"C{str(i).zfill(3)}" for i in range(1, len(df_coords) + 1)
]

df_coords = df_coords[['city_id', 'city', 'lat', 'lon']]
print(df_coords.head())
df_coords.tail()

  city_id               city         lat         lon
0    C001  Mont Saint Michel  48.6359541  -1.5114600
1    C002         Saint Malo  48.6495180  -2.0260409
2    C003             Bayeux  49.2764624  -0.7024738
3    C004           Le Havre  49.4938975   0.1079732
4    C005              Rouen  49.4404591   1.0939658


,city_id,city,lat,lon
30,C031,Toulouse,43.6044638,1.4442433
31,C032,Montauban,44.0175835,1.3549991
32,C033,Biarritz,43.4711438,-1.5527266
33,C034,Bayonne,43.4945144,-1.4736657
34,C035,La Rochelle,46.1597320,-1.1515951


### Store into csv file before load it in S3 bucket

In [13]:
df_coords.to_csv("../data/raw/cities_coords.csv", index=False)

In [14]:
upload_file_to_s3("../data/raw/cities_coords.csv", folder="raw")

Uploaded cities_coords.csv to s3://fullstack-certification/kayak/raw/cities_coords.csv


## 7 days weather forecast for each city using Open-meteo API

In [11]:
df_coords = pd.read_csv("../data/raw/cities_coords.csv")
df_coords.head()

,city_id,city,lat,lon
0,C001,Mont Saint Michel,48.635954,-1.511460
1,C002,Saint Malo,48.649518,-2.026041
2,C003,Bayeux,49.276462,-0.702474
3,C004,Le Havre,49.493898,0.107973
4,C005,Rouen,49.440459,1.093966


In [16]:
url = "https://api.open-meteo.com/v1/forecast"

daily_vars = ",".join([
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "sunshine_duration",
    "windspeed_10m_max",
    "relative_humidity_2m_mean"
])

city = df_coords["city"][0]
lat = df_coords["lat"][0]
lon = df_coords["lon"][0]

params = {
    "latitude": lat,
    "longitude": lon,
    "daily": daily_vars,
    "forecast_day": 7,
    "timezone": "Europe/Paris"
}

response = requests.get(url, params=params, timeout=15)
print(response.raise_for_status())
data = response.json()
data

None


{'latitude': 48.64,
 'longitude': -1.5100002,
 'generationtime_ms': 1.7946958541870117,
 'utc_offset_seconds': 3600,
 'timezone': 'Europe/Paris',
 'timezone_abbreviation': 'GMT+1',
 'elevation': 31.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm',
  'sunshine_duration': 's',
  'windspeed_10m_max': 'km/h',
  'relative_humidity_2m_mean': '%'},
 'daily': {'time': ['2025-10-28',
   '2025-10-29',
   '2025-10-30',
   '2025-10-31',
   '2025-11-01',
   '2025-11-02',
   '2025-11-03'],
  'temperature_2m_max': [14.9, 13.4, 13.3, 18.0, 14.7, 14.1, 16.7],
  'temperature_2m_min': [12.0, 11.2, 7.3, 11.8, 9.6, 9.5, 12.2],
  'precipitation_sum': [0.0, 3.8, 0.0, 0.7, 10.8, 0.0, 0.0],
  'sunshine_duration': [10562.17,
   1260.74,
   24407.64,
   18000.0,
   18301.2,
   30631.09,
   13347.88],
  'windspeed_10m_max': [29.8, 35.4, 32.4, 34.8, 35.0, 29.2, 30.5],
  'relative_humidity_2m_mean': [85, 89, 80, 87, 87, 83, 87]}}

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"

daily_vars = ",".join([
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "sunshine_duration",
    "windspeed_10m_max",
    "relative_humidity_2m_mean"
])

weather = []

for _, row in df_coords.iterrows():
    city = row["city"]
    lat  = float(row["lat"])
    lon  = float(row["lon"])

    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": daily_vars,
        "forecast_days": 7,
        "timezone": "Europe/Paris"
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        daily = data.get("daily", {})
        dates = daily.get("time", [])
        
        tmax = daily.get("temperature_2m_max", [])
        tmin = daily.get("temperature_2m_min", [])
        prcp = daily.get("precipitation_sum", [])
        sunsec = daily.get("sunshine_duration", [])
        wmax = daily.get("windspeed_10m_max", [])
        rhmean = daily.get("relative_humidity_2m_mean", [])

        for i in range(len(dates)):
            weather.append({
                "city": city,
                "date": dates[i],
                "lat": lat,
                "lon": lon,
                "temp_max_c": tmax[i]   if i < len(tmax) else None,
                "temp_min_c": tmin[i]   if i < len(tmin) else None,
                "precip_mm":  prcp[i]   if i < len(prcp) else None,
                "sunshine_h": round(sunsec[i] / 3600, 2) if i < len(sunsec) and sunsec[i] is not None else None,  # Convert sunshine duration (seconds → hours)
                "wind_max_kmh": round(wmax[i] * 3.6, 2) if i < len(wmax) and wmax[i] is not None else None,       # Convert wind speed (m/s → km/h)
                "rh_mean_pct": rhmean[i] if i < len(rhmean) else None
                })
        
        time.sleep(0.5)
        print(f"{city}: fetched 7-day forecast")

    except Exception as e:
        print(f"Error for {city}: {e}")
        time.sleep(0.5)

Mont Saint Michel: fetched 7-day forecast
Saint Malo: fetched 7-day forecast
Bayeux: fetched 7-day forecast
Le Havre: fetched 7-day forecast
Rouen: fetched 7-day forecast
Paris: fetched 7-day forecast
Amiens: fetched 7-day forecast
Lille: fetched 7-day forecast
Strasbourg: fetched 7-day forecast
Chateau du Haut Koenigsbourg: fetched 7-day forecast
Colmar: fetched 7-day forecast
Eguisheim: fetched 7-day forecast
Besancon: fetched 7-day forecast
Dijon: fetched 7-day forecast
Annecy: fetched 7-day forecast
Grenoble: fetched 7-day forecast
Lyon: fetched 7-day forecast
Gorges du Verdon: fetched 7-day forecast
Bormes les Mimosas: fetched 7-day forecast
Cassis: fetched 7-day forecast
Marseille: fetched 7-day forecast
Aix en Provence: fetched 7-day forecast
Avignon: fetched 7-day forecast
Uzes: fetched 7-day forecast
Nimes: fetched 7-day forecast
Aigues Mortes: fetched 7-day forecast
Saintes Maries de la mer: fetched 7-day forecast
Collioure: fetched 7-day forecast
Carcassonne: fetched 7-day f

In [18]:
df_weather_daily_7d = pd.DataFrame(weather)
df_weather_daily_7d.head()

,city,date,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct
0,Mont Saint Michel,2025-10-28,48.635954,-1.51146,14.9,12.0,0.0,2.93,107.28,85
1,Mont Saint Michel,2025-10-29,48.635954,-1.51146,13.4,11.2,3.8,0.35,127.44,89
2,Mont Saint Michel,2025-10-30,48.635954,-1.51146,13.3,7.3,0.0,6.78,116.64,80
3,Mont Saint Michel,2025-10-31,48.635954,-1.51146,18.0,11.8,0.7,5.00,125.28,87
4,Mont Saint Michel,2025-11-01,48.635954,-1.51146,14.7,9.6,10.8,5.08,126.00,87


In [19]:
len(df_weather_daily_7d)

245

#### Merged city_id and create a weather_id

In [20]:
df_weather_daily_7d = df_weather_daily_7d.merge(
    right=df_coords[["city", "city_id"]],
    on='city',
    how='left'
)

df_weather_daily_7d.head()

,city,date,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct,city_id
0,Mont Saint Michel,2025-10-28,48.635954,-1.51146,14.9,12.0,0.0,2.93,107.28,85,C001
1,Mont Saint Michel,2025-10-29,48.635954,-1.51146,13.4,11.2,3.8,0.35,127.44,89,C001
2,Mont Saint Michel,2025-10-30,48.635954,-1.51146,13.3,7.3,0.0,6.78,116.64,80,C001
3,Mont Saint Michel,2025-10-31,48.635954,-1.51146,18.0,11.8,0.7,5.00,125.28,87,C001
4,Mont Saint Michel,2025-11-01,48.635954,-1.51146,14.7,9.6,10.8,5.08,126.00,87,C001


In [21]:
df_weather_daily_7d.columns

Index(['city', 'date', 'lat', 'lon', 'temp_max_c', 'temp_min_c', 'precip_mm',
       'sunshine_h', 'wind_max_kmh', 'rh_mean_pct', 'city_id'],
      dtype='object')

In [22]:
df_weather_daily_7d["weather_id"] = [
    f"W{str(i).zfill(3)}" for i in range(1, len(df_weather_daily_7d) + 1)
]

df_weather_daily_7d = df_weather_daily_7d[['weather_id', 'city_id', 'city', 'date', 'lat', 
                                          'lon', 'temp_max_c', 'temp_min_c', 'precip_mm',
                                          'sunshine_h', 'wind_max_kmh', 'rh_mean_pct']]

print(df_weather_daily_7d.head())
df_weather_daily_7d.tail()

  weather_id city_id               city        date        lat      lon  \
0       W001    C001  Mont Saint Michel  2025-10-28  48.635954 -1.51146   
1       W002    C001  Mont Saint Michel  2025-10-29  48.635954 -1.51146   
2       W003    C001  Mont Saint Michel  2025-10-30  48.635954 -1.51146   
3       W004    C001  Mont Saint Michel  2025-10-31  48.635954 -1.51146   
4       W005    C001  Mont Saint Michel  2025-11-01  48.635954 -1.51146   

   temp_max_c  temp_min_c  precip_mm  sunshine_h  wind_max_kmh  rh_mean_pct  
0        14.9        12.0        0.0        2.93        107.28           85  
1        13.4        11.2        3.8        0.35        127.44           89  
2        13.3         7.3        0.0        6.78        116.64           80  
3        18.0        11.8        0.7        5.00        125.28           87  
4        14.7         9.6       10.8        5.08        126.00           87  


,weather_id,city_id,city,date,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct
240,W241,C035,La Rochelle,2025-10-30,46.159732,-1.151595,16.6,9.1,0.0,8.76,70.92,87
241,W242,C035,La Rochelle,2025-10-31,46.159732,-1.151595,17.4,12.7,0.3,4.48,127.80,90
242,W243,C035,La Rochelle,2025-11-01,46.159732,-1.151595,16.2,13.6,12.1,6.46,135.72,83
243,W244,C035,La Rochelle,2025-11-02,46.159732,-1.151595,15.5,14.6,3.9,3.33,134.28,81
244,W245,C035,La Rochelle,2025-11-03,46.159732,-1.151595,16.3,14.4,0.0,8.91,65.52,86


### Store into csv file before load it in S3 bucket

In [23]:
df_weather_daily_7d.to_csv("../data/raw/weather_daily_7d.csv", index=False)

In [24]:
upload_file_to_s3("../data/raw/weather_daily_7d.csv", folder="raw")

Uploaded weather_daily_7d.csv to s3://fullstack-certification/kayak/raw/weather_daily_7d.csv


### Create a weather score to see the best places to go

In [ ]:
df_weather_daily_7d = pd.read_csv("../data/raw/weather_daily_7d.csv")
df_weather_daily_7d.head()

,weather_id,city_id,city,date,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct
0,W001,C001,Mont Saint Michel,2025-10-28,48.635954,-1.51146,14.9,12.0,0.0,2.93,107.28,85
1,W002,C001,Mont Saint Michel,2025-10-29,48.635954,-1.51146,13.4,11.2,3.8,0.35,127.44,89
2,W003,C001,Mont Saint Michel,2025-10-30,48.635954,-1.51146,13.3,7.3,0.0,6.78,116.64,80
3,W004,C001,Mont Saint Michel,2025-10-31,48.635954,-1.51146,18.0,11.8,0.7,5.00,125.28,87
4,W005,C001,Mont Saint Michel,2025-11-01,48.635954,-1.51146,14.7,9.6,10.8,5.08,126.00,87


In [26]:
df_weather_daily_7d.isna().sum()

weather_id      0
city_id         0
city            0
date            0
lat             0
lon             0
temp_max_c      0
temp_min_c      0
precip_mm       0
sunshine_h      0
wind_max_kmh    0
rh_mean_pct     0
dtype: int64

In [27]:
df_weather_daily_7d.columns

Index(['weather_id', 'city_id', 'city', 'date', 'lat', 'lon', 'temp_max_c',
       'temp_min_c', 'precip_mm', 'sunshine_h', 'wind_max_kmh', 'rh_mean_pct'],
      dtype='object')

In [28]:
df_avg = (
    df_weather_daily_7d
    .groupby(["city_id","city", "lat", "lon"])[
        ["temp_max_c", "temp_min_c", "precip_mm",
         "sunshine_h", "wind_max_kmh", "rh_mean_pct"]
    ]
    .mean()
    .round(2)
    .reset_index()
)

df_avg["temp_mean"] = round((df_avg["temp_max_c"] + df_avg["temp_min_c"]) / 2, 2)
df_avg.head()

,city_id,city,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct,temp_mean
0,C001,Mont Saint Michel,48.635954,-1.511460,15.01,10.51,2.19,4.62,116.79,85.43,12.76
1,C002,Saint Malo,48.649518,-2.026041,15.27,10.90,0.57,4.98,114.74,83.71,13.08
2,C003,Bayeux,49.276462,-0.702474,14.60,10.09,2.59,4.19,98.43,87.00,12.34
3,C004,Le Havre,49.493898,0.107973,14.96,11.69,2.93,3.68,112.01,82.43,13.32
4,C005,Rouen,49.440459,1.093966,14.91,10.10,2.61,3.05,69.17,85.29,12.50


In [29]:
len(df_avg)

35

#### Compute a weather score with tunable weights

In [ ]:
df_avg["weather_score"] = (
    0.4 * df_avg["temp_mean"]              # warmer is nicer 
    - 0.3 * df_avg["precip_mm"]            # less rain is better
    + 0.2 * (df_avg["sunshine_h"] / 10)    # more sun is better (scaled)
    - 0.1 * (df_avg["rh_mean_pct"] / 10)   # very humid feels worse (scaled)
    - 0.05 * (df_avg["wind_max_kmh"] / 10) # strong wind is less comfortable
).round(2)

df_avg.head()

,city_id,city,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct,temp_mean,weather_score
0,C001,Mont Saint Michel,48.635954,-1.511460,15.01,10.51,2.19,4.62,116.79,85.43,12.76,3.10
1,C002,Saint Malo,48.649518,-2.026041,15.27,10.90,0.57,4.98,114.74,83.71,13.08,3.75
2,C003,Bayeux,49.276462,-0.702474,14.60,10.09,2.59,4.19,98.43,87.00,12.34,2.88
3,C004,Le Havre,49.493898,0.107973,14.96,11.69,2.93,3.68,112.01,82.43,13.32,3.14
4,C005,Rouen,49.440459,1.093966,14.91,10.10,2.61,3.05,69.17,85.29,12.50,3.08


#### Normalize the score to get a better vision

In [31]:
min_score = df_avg["weather_score"].min()
max_score = df_avg["weather_score"].max()

df_avg["weather_score_norm"] = (
    10 * (df_avg["weather_score"] - min_score) / (max_score - min_score)
).round(2)

df_avg.head()

,city_id,city,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct,temp_mean,weather_score,weather_score_norm
0,C001,Mont Saint Michel,48.635954,-1.511460,15.01,10.51,2.19,4.62,116.79,85.43,12.76,3.10,3.68
1,C002,Saint Malo,48.649518,-2.026041,15.27,10.90,0.57,4.98,114.74,83.71,13.08,3.75,5.31
2,C003,Bayeux,49.276462,-0.702474,14.60,10.09,2.59,4.19,98.43,87.00,12.34,2.88,3.13
3,C004,Le Havre,49.493898,0.107973,14.96,11.69,2.93,3.68,112.01,82.43,13.32,3.14,3.78
4,C005,Rouen,49.440459,1.093966,14.91,10.10,2.61,3.05,69.17,85.29,12.50,3.08,3.63


In [33]:
df_avg.to_csv("../data/processed/weather_avg_score.csv", index=False)

In [34]:
upload_file_to_s3("../data/processed/weather_avg_score.csv", folder="processed")

Uploaded weather_avg_score.csv to s3://fullstack-certification/kayak/processed/weather_avg_score.csv


In [35]:
top5 = df_avg.sort_values("weather_score_norm", axis=0, ascending=False).head(5)

top5

,city_id,city,lat,lon,temp_max_c,temp_min_c,precip_mm,sunshine_h,wind_max_kmh,rh_mean_pct,temp_mean,weather_score,weather_score_norm
20,C021,Marseille,43.296174,5.369953,20.20,14.76,1.01,7.62,88.71,77.71,17.48,5.62,10.00
19,C020,Cassis,43.214036,5.539632,20.61,14.81,1.84,7.59,82.13,77.14,17.71,5.50,9.70
27,C028,Collioure,42.525050,3.083155,19.94,12.41,0.04,6.76,78.27,72.57,16.18,5.48,9.65
25,C026,Aigues Mortes,43.566152,4.191540,20.24,13.17,0.97,7.05,82.03,78.71,16.70,5.33,9.27
28,C029,Carcassonne,43.213036,2.349107,19.56,12.37,0.01,6.34,84.81,78.86,15.96,5.30,9.20


In [36]:
top5.to_csv("../data/processed/top5.csv", index=False)

In [37]:
upload_file_to_s3("../data/processed/top5.csv", folder="processed")

Uploaded top5.csv to s3://fullstack-certification/kayak/processed/top5.csv


### Plot the 20 cities and the 5 best destinations

In [ ]:
fig = px.scatter_map(
    df_avg,
    lat="lat",
    lon="lon",
    hover_name="city",
    hover_data={"weather_score_norm": True},
    color="weather_score_norm",
    size="weather_score_norm",
    color_continuous_scale="Bluered",   
    size_max=18,
    zoom=4.5,                          # adjust zoom (4.5–5 for France)
    center={"lat": 46.5, "lon": 2.5},  # center on France
    opacity=0.85,
    title="Cities — Weather Score"
)

fig.update_layout(
    mapbox_style="carto-positron",
    width=900,     
    height=650,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.update_traces(marker=dict(sizemin=5))
fig.show()

In [39]:
fig.write_html("../charts/cities_weather_score.html")

In [40]:
fig = px.scatter_map(
    top5,
    lat="lat",
    lon="lon",
    hover_name="city",
    hover_data={"weather_score_norm": True},
    color="weather_score_norm",
    size="weather_score_norm",
    color_continuous_scale="Bluered",   
    # range_color=[0, 10],
    size_max=18,
    zoom=4.5,
    opacity=0.85,                          
    center={"lat": 46.5, "lon": 2.5},  
    title="Top 5 Destinations — Weather Score"
)

fig.update_layout(
    mapbox_style="carto-positron",
    width=900,     
    height=650,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.show()

In [41]:
fig.write_html("../charts/top_5_cities_weather_score.html")

### Scraping Booking

In [3]:
top5 = pd.read_csv("../data/processed/top5.csv")

In [4]:
for city in top5['city']:
    print(city)

Marseille
Cassis
Collioure
Aigues Mortes
Carcassonne


In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

def get_hotel_details(hotel_url, max_retries=3):
    """Extract lat/lon and description with automatic retry."""
    
    for attempt in range(max_retries):
        try:
            r = requests.get(hotel_url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            
            # Lat/lon
            lat, lon = None, None
            
            # Main lookup
            latlon_elem = soup.find("a", {"class": "show_map"})
            if latlon_elem and latlon_elem.get('data-atlas-latlng'):
                try:
                    lat, lon = latlon_elem.get('data-atlas-latlng').split(",")
                    lat, lon = float(lat.strip()), float(lon.strip())
                except:
                    pass
            
            # Fallback lookup
            if not lat or not lon:
                latlon_elem = soup.select_one("a[data-atlas-latlng]")
                if latlon_elem and latlon_elem.has_attr("data-atlas-latlng"):
                    try:
                        lat, lon = latlon_elem["data-atlas-latlng"].split(",")
                        lat, lon = float(lat.strip()), float(lon.strip())
                    except:
                        pass
            
            # Description
            description = None
            
            # Main lookup
            description_elem = soup.find("div", {"class": "hp_desc_main_content"})
            if description_elem:
                description = description_elem.get_text(strip=True)
            
            # Fallback lookup
            if not description:
                desc_selectors = [
                    "p[data-testid='property-description']",
                    "div[data-testid='property-description']",
                    "div#property_description_content p"
                ]
                for selector in desc_selectors:
                    desc_elem = soup.select_one(selector)
                    if desc_elem:
                        description = desc_elem.get_text(strip=True)
                        break
            
            if lat and lon and description:
                return lat, lon, description
            
            if attempt < max_retries - 1:
                time.sleep(random.uniform(10, 20))
            else:
                return lat, lon, description
            
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Error getting details for {hotel_url}: {e}")
            else:
                time.sleep(random.uniform(10, 20))
    
    return None, None, None

def scrape_hotels_for_city(city, max_hotels=20):
    """
    Scrape up to `max_hotels` hotels for a given city on Booking.com.
    Returns a list of hotel dictionaries.
    """
    hotels = []
    url = f"https://www.booking.com/searchresults.html?ss={city.replace(' ', '+')}&rows={max_hotels}"
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        cards = soup.select("div[data-testid='property-card']")
        
        for i, card in enumerate(cards[:max_hotels], 1):
            name = card.select_one("div[data-testid='title']")
            rating = card.select_one("div.f63b14ab7a.dff2e52086")
            link = card.select_one("a[data-testid='title-link']")
            
            hotel_url = None
            if link and link.has_attr("href"):
                full_url = link["href"]
                if not full_url.startswith("http"):
                    full_url = "https://www.booking.com" + full_url
                hotel_url = full_url.split("?")[0]
            
            lat, lon, description = (None, None, None)
            if hotel_url:
                lat, lon, description = get_hotel_details(hotel_url, max_retries=3)
                time.sleep(random.uniform(7, 20))
            
            hotels.append({
                "city": city,
                "hotel_name": name.text.strip() if name else None,
                "user_rating": rating.text.strip() if rating else None,
                "hotel_url": hotel_url,
                "description": description,
                "lat": lat,
                "lon": lon,
            })
        
        print(f"{len(hotels)} hotels scraped for {city}")
        
    except Exception as e:
        print(f"Error scraping {city}: {e}")
    
    time.sleep(random.uniform(7, 20))
    return hotels

hotel = []
city_hotel = scrape_hotels_for_city('Marseille', max_hotels=20)
hotel.extend(city_hotel)

20 hotels scraped for Marseille


In [40]:
hotel

[{'city': 'Marseille',
  'hotel_name': 'InterContinental Marseille - Hotel Dieu by IHG',
  'user_rating': '8,7',
  'hotel_url': 'https://www.booking.com/hotel/fr/marseille-dieu.fr.html',
  'description': "Bénéficiez d'un traitement VIP grâce au service exclusif de l'établissement InterContinental Marseille - Hotel Dieu by IHGL'InterContinental Marseille\xa0-\xa0Hotel Dieu, an IHG Hotel vous accueille dans un magnifique bâtiment datant du XVIIIe siècle, dans le quartier historique de Marseille, à 350\xa0mètres du Vieux-Port. Une connexion Wi-Fi est disponible.\n\nProposant 194\xa0chambres et suites, dont certaines offrant une vue sur les sites d'intérêt et dotées d'une terrasse privée, cet hôtel est décoré dans un style contemporain et élégant en harmonie avec son cachet historique.\n\nLa Brasserie Les Fenêtres. Le bar Capian propose de nombreux cocktails préparés par des experts.\n\nL’espace détente de l’hôtel comporte un spa Clarins avec des saunas, des hammams, des lits de bronzage, 

In [ ]:
def complete_missing_data(hotels):
    """Complete missing description, lat, lon data."""
    updated_count = 0
    
    for i, hotel in enumerate(hotels):
        missing_desc = hotel['description'] is None
        missing_coords = hotel['lat'] is None or hotel['lon'] is None
        
        if (missing_desc or missing_coords) and hotel['hotel_url']:
            print(f"[{i+1}/{len(hotels)}] {hotel['hotel_name']}, {hotel['city']}")
            
            lat, lon, description = get_hotel_details(hotel['hotel_url'], max_retries=3)
            
            if missing_desc and description:
                hotel['description'] = description
            
            if missing_coords and lat and lon:
                hotel['lat'] = lat
                hotel['lon'] = lon
            
            updated_count += 1
            time.sleep(random.uniform(15, 25))
    
    print(f"\nCompleted: {updated_count} hotels updated")
    return hotels

hotel_updated = complete_missing_data(hotel)

[17/20] Osea - A 50 mètres du Vieux-Port, Marseille

Completed: 1 hotels updated


In [43]:
hotel_updated

[{'city': 'Marseille',
  'hotel_name': 'InterContinental Marseille - Hotel Dieu by IHG',
  'user_rating': '8,7',
  'hotel_url': 'https://www.booking.com/hotel/fr/marseille-dieu.fr.html',
  'description': "Bénéficiez d'un traitement VIP grâce au service exclusif de l'établissement InterContinental Marseille - Hotel Dieu by IHGL'InterContinental Marseille\xa0-\xa0Hotel Dieu, an IHG Hotel vous accueille dans un magnifique bâtiment datant du XVIIIe siècle, dans le quartier historique de Marseille, à 350\xa0mètres du Vieux-Port. Une connexion Wi-Fi est disponible.\n\nProposant 194\xa0chambres et suites, dont certaines offrant une vue sur les sites d'intérêt et dotées d'une terrasse privée, cet hôtel est décoré dans un style contemporain et élégant en harmonie avec son cachet historique.\n\nLa Brasserie Les Fenêtres. Le bar Capian propose de nombreux cocktails préparés par des experts.\n\nL’espace détente de l’hôtel comporte un spa Clarins avec des saunas, des hammams, des lits de bronzage, 

In [44]:
all_hotels = []
for city in top5['city']:
    city_hotels = scrape_hotels_for_city(city, max_hotels=20)
    all_hotels.extend(city_hotels)

20 hotels scraped for Marseille
20 hotels scraped for Cassis
20 hotels scraped for Collioure
20 hotels scraped for Aigues Mortes
20 hotels scraped for Carcassonne


In [45]:
all_hotels

[{'city': 'Marseille',
  'hotel_name': 'Radisson Blu Hotel Marseille Vieux Port',
  'user_rating': '8,6',
  'hotel_url': 'https://www.booking.com/hotel/fr/radisson-sas-marseille-vieux-port.fr.html',
  'description': "Vous pouvez peut-être bénéficier d’une réduction Genius dans l’établissement Radisson Blu Hotel Marseille Vieux Port.Connectez-vouspour voir si une réduction Genius est disponible à vos dates.Les réductions Genius proposées par cet établissement dépendent des dates de réservation, des dates de séjour et des autres offres disponibles.Surplombant le Vieux-Port, le Radisson Blu Hotel Marseille Vieux Port vous accueille à Marseille. Cet hôtel 4 étoiles possède une piscine extérieure donnant sur le fort Saint-Nicolas et une salle de sport offrant une vue panoramique sur les principaux monuments de la ville. Une connexion Wi-Fi est disponible gratuitement.\n\nLes chambres sont climatisées et décorées dans un style provençal ou africain. Certaines offrent une vue sur le port et s

In [46]:
hotel_updated = complete_missing_data(all_hotels)

[10/100] New Hotel Le Quai - Vieux Port, Marseille
[12/100] Les Appartements du Vieux Port, Marseille
[13/100] Novotel Marseille Vieux Port, Marseille
[21/100] HPC Suites - Cassis Centre - Parking Gratuit & Piscine Chauffée, Cassis
[35/100] Le Best à Cassis,Top emplacement,Parking,double balcon, Cassis
[43/100] Les Roches Brunes, Collioure
[60/100] Apartment Olive & Kin, Collioure, Collioure
[65/100] Hôtel les jardins du canal - piscine chauffée - parking - garage - bornes électriques, Aigues Mortes
[74/100] Boutique Hôtel des Remparts & Spa, Aigues Mortes
[75/100] Les Appartements du Mas - Rose Marie, Aigues Mortes
[78/100] La maison sur la place dans Aigues Mortes, Aigues Mortes
[79/100] Maison des Croisades, Aigues Mortes
[88/100] Le Jardin de La Tour Pinte, Carcassonne
[96/100] B&B Bloc G, Carcassonne

Completed: 14 hotels updated


In [50]:
hotels_updated = complete_missing_data(hotel_updated)

[12/100] Les Appartements du Vieux Port, Marseille

Completed: 1 hotels updated


In [52]:
df_hotels = pd.DataFrame(hotels_updated)
df_hotels.head()

,city,hotel_name,user_rating,hotel_url,description,lat,lon
0,Marseille,Radisson Blu Hotel Marseille Vieux Port,"8,6",https://www.booking.com/hotel/fr/radisson-sas-...,Vous pouvez peut-être bénéficier d’une réducti...,43.292583,5.367028
1,Marseille,Golden Tulip Marseille Euromed,"7,7",https://www.booking.com/hotel/fr/golden-tulip....,Vous pouvez peut-être bénéficier d’une réducti...,43.309722,5.367034
2,Marseille,Maisons du Monde Hôtel & Suites - Marseille Vi...,"8,7",https://www.booking.com/hotel/fr/maisondumonde...,Vous pouvez peut-être bénéficier d’une réducti...,43.294124,5.374640
3,Marseille,InterContinental Marseille - Hotel Dieu by IHG,"8,7",https://www.booking.com/hotel/fr/marseille-die...,Bénéficiez d'un traitement VIP grâce au servic...,43.298582,5.369746
4,Marseille,The Babel Community Hôtel - Vieux Port,"8,3",https://www.booking.com/hotel/fr/grand-t1-vieu...,Vous pouvez peut-être bénéficier d’une réducti...,43.293758,5.377448


In [54]:
df_hotels.isna().sum()

city           0
hotel_name     0
user_rating    0
hotel_url      0
description    0
lat            0
lon            0
dtype: int64

In [55]:
df_hotels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   city         100 non-null    object 
 1   hotel_name   100 non-null    object 
 2   user_rating  100 non-null    object 
 3   hotel_url    100 non-null    object 
 4   description  100 non-null    object 
 5   lat          100 non-null    float64
 6   lon          100 non-null    float64
dtypes: float64(2), object(5)
memory usage: 5.6+ KB


In [56]:
df_hotels["user_rating"] = (
    df_hotels["user_rating"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [57]:
type(df_hotels["user_rating"][0])

numpy.float64

In [60]:
df_hotels = df_hotels.merge(
    right=df_coords[["city", "city_id"]],
    on="city",
    how='left'
)

df_hotels.head()

,city,hotel_name,user_rating,hotel_url,description,lat,lon,city_id
0,Marseille,Radisson Blu Hotel Marseille Vieux Port,8.6,https://www.booking.com/hotel/fr/radisson-sas-...,Vous pouvez peut-être bénéficier d’une réducti...,43.292583,5.367028,C021
1,Marseille,Golden Tulip Marseille Euromed,7.7,https://www.booking.com/hotel/fr/golden-tulip....,Vous pouvez peut-être bénéficier d’une réducti...,43.309722,5.367034,C021
2,Marseille,Maisons du Monde Hôtel & Suites - Marseille Vi...,8.7,https://www.booking.com/hotel/fr/maisondumonde...,Vous pouvez peut-être bénéficier d’une réducti...,43.294124,5.374640,C021
3,Marseille,InterContinental Marseille - Hotel Dieu by IHG,8.7,https://www.booking.com/hotel/fr/marseille-die...,Bénéficiez d'un traitement VIP grâce au servic...,43.298582,5.369746,C021
4,Marseille,The Babel Community Hôtel - Vieux Port,8.3,https://www.booking.com/hotel/fr/grand-t1-vieu...,Vous pouvez peut-être bénéficier d’une réducti...,43.293758,5.377448,C021


In [61]:
df_hotels.columns

Index(['city', 'hotel_name', 'user_rating', 'hotel_url', 'description', 'lat',
       'lon', 'city_id'],
      dtype='object')

In [62]:
df_hotels["hotel_id"] = [
    f"H{str(i).zfill(3)}" for i in range(1, len(df_hotels) + 1)
]

df_hotels = df_hotels[['hotel_id', 'city_id', 'city', 'hotel_name', 
                      'user_rating', 'hotel_url', 'description', 'lat','lon']]

print(df_hotels.head())
df_hotels.tail()

  hotel_id city_id       city  \
0     H001    C021  Marseille   
1     H002    C021  Marseille   
2     H003    C021  Marseille   
3     H004    C021  Marseille   
4     H005    C021  Marseille   

                                          hotel_name  user_rating  \
0            Radisson Blu Hotel Marseille Vieux Port          8.6   
1                     Golden Tulip Marseille Euromed          7.7   
2  Maisons du Monde Hôtel & Suites - Marseille Vi...          8.7   
3     InterContinental Marseille - Hotel Dieu by IHG          8.7   
4             The Babel Community Hôtel - Vieux Port          8.3   

                                           hotel_url  \
0  https://www.booking.com/hotel/fr/radisson-sas-...   
1  https://www.booking.com/hotel/fr/golden-tulip....   
2  https://www.booking.com/hotel/fr/maisondumonde...   
3  https://www.booking.com/hotel/fr/marseille-die...   
4  https://www.booking.com/hotel/fr/grand-t1-vieu...   

                                         descript

,hotel_id,city_id,city,hotel_name,user_rating,hotel_url,description,lat,lon
95,H096,C029,Carcassonne,B&B Bloc G,8.5,https://www.booking.com/hotel/fr/bloc-g.fr.html,Ce Bed & Breakfast design se trouve à 5 minute...,43.209735,2.360811
96,H097,C029,Carcassonne,Chambres d'hotes - Le Magnolia,9.5,https://www.booking.com/hotel/fr/b-amp-b-le-ma...,L’établissement Chambres d'hotes - Le Magnolia...,43.215548,2.355568
97,H098,C029,Carcassonne,"Les Gîtes de Laure - O Lit Divin, Instant de v...",9.2,https://www.booking.com/hotel/fr/o-lit-divin-i...,Vous pouvez peut-être bénéficier d’une réducti...,43.207437,2.361331
98,H099,C029,Carcassonne,Longue Vie Au Roi - Vue Enchanteresse,9.1,https://www.booking.com/hotel/fr/longue-vie-au...,Vous pouvez peut-être bénéficier d’une réducti...,43.206080,2.361138
99,H100,C029,Carcassonne,Situation exceptionnelle HARMONY III,9.8,https://www.booking.com/hotel/fr/harmony-3.fr....,Vous pouvez peut-être bénéficier d’une réducti...,43.211814,2.352205


In [63]:
df_hotels.to_csv("../data/raw/all_hotels.csv", index=False)

In [66]:
upload_file_to_s3("../data/raw/all_hotels.csv", folder="raw")

Uploaded all_hotels.csv to s3://fullstack-certification/kayak/raw/all_hotels.csv


In [9]:
df_hotels = pd.read_csv("../data/raw/all_hotels.csv")
df_hotels.head()

,hotel_id,city_id,city,hotel_name,user_rating,hotel_url,description,lat,lon
0,H001,C021,Marseille,Radisson Blu Hotel Marseille Vieux Port,8.6,https://www.booking.com/hotel/fr/radisson-sas-...,Vous pouvez peut-être bénéficier d’une réducti...,43.292583,5.367028
1,H002,C021,Marseille,Golden Tulip Marseille Euromed,7.7,https://www.booking.com/hotel/fr/golden-tulip....,Vous pouvez peut-être bénéficier d’une réducti...,43.309722,5.367034
2,H003,C021,Marseille,Maisons du Monde Hôtel & Suites - Marseille Vi...,8.7,https://www.booking.com/hotel/fr/maisondumonde...,Vous pouvez peut-être bénéficier d’une réducti...,43.294124,5.374640
3,H004,C021,Marseille,InterContinental Marseille - Hotel Dieu by IHG,8.7,https://www.booking.com/hotel/fr/marseille-die...,Bénéficiez d'un traitement VIP grâce au servic...,43.298582,5.369746
4,H005,C021,Marseille,The Babel Community Hôtel - Vieux Port,8.3,https://www.booking.com/hotel/fr/grand-t1-vieu...,Vous pouvez peut-être bénéficier d’une réducti...,43.293758,5.377448


In [68]:
len(df_hotels)

100

In [69]:
df_best_hotels = df_hotels.sort_values("user_rating", axis=0, ascending=False).head(20)

df_best_hotels.head()

,hotel_id,city_id,city,hotel_name,user_rating,hotel_url,description,lat,lon
22,H023,C020,Cassis,Chambre d'hôtes Clos du Petit Jésus,10.0,https://www.booking.com/hotel/fr/clos-du-petit...,"Situé à Cassis, à seulement 5 minutes à pied d...",43.215246,5.544668
86,H087,C029,Carcassonne,"Le Petit Chai niché sous la Cité,",9.9,https://www.booking.com/hotel/fr/le-chai-niche...,Vous pouvez peut-être bénéficier d’une réducti...,43.205759,2.360957
82,H083,C029,Carcassonne,La Maison de La Tour Pinte,9.9,https://www.booking.com/hotel/fr/la-maison-de-...,Vous pouvez peut-être bénéficier d’une réducti...,43.205804,2.360999
67,H068,C026,Aigues Mortes,La Perle Rose Villa Standing 5 étoiles avec pi...,9.9,https://www.booking.com/hotel/fr/la-perle-rose...,Vous pouvez peut-être bénéficier d’une réducti...,43.562119,4.185999
85,H086,C029,Carcassonne,Le Loft de La Tour Pinte,9.8,https://www.booking.com/hotel/fr/le-loft-de-la...,Vous pouvez peut-être bénéficier d’une réducti...,43.207188,2.361495


In [70]:
df_best_hotels.to_csv("../data/processed/best_hotels.csv", index=False)

In [71]:
upload_file_to_s3("../data/processed/best_hotels.csv", folder="processed")

Uploaded best_hotels.csv to s3://fullstack-certification/kayak/processed/best_hotels.csv


In [8]:
df_best_hotels = pd.read_csv("../data/processed/best_hotels.csv")

In [73]:
fig = px.scatter_map(
    df_best_hotels,
    lat="lat",
    lon="lon",
    hover_name="hotel_name",
    hover_data={"user_rating": True},
    color="user_rating",
    size="user_rating",
    color_continuous_scale="Bluered",   
    # range_color=[0, 10],
    size_max=18,
    zoom=4.5,                          
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.85,  
    title="Top 20 Hotels in France's 5 Best Weather Cities"
)

fig.update_layout(
    mapbox_style="carto-positron",
    width=900,     
    height=650,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.show()


In [75]:
fig.write_html("../charts/best_hotels.html")

#### Create Database on NeonDB

In [3]:
load_dotenv()

engine = create_engine(os.getenv("NEON_DB_URL"), pool_pre_ping=True)
engine.connect()

In [3]:
def run_sql(sql):
    """Execute a SQL query on the connected database."""
    with engine.begin() as conn:
        conn.execute(text(sql))

In [5]:
run_sql("""
CREATE TABLE IF NOT EXISTS city (
  city_id   VARCHAR(8)  PRIMARY KEY,
  city      TEXT        NOT NULL,
  lat       DOUBLE PRECISION NOT NULL,
  lon       DOUBLE PRECISION NOT NULL
);
""")

In [6]:
run_sql("""
CREATE TABLE IF NOT EXISTS weather (
  weather_id     VARCHAR(8) PRIMARY KEY,
  city_id        VARCHAR(8) NOT NULL,
  date           DATE NOT NULL,
  temp_max_c     DOUBLE PRECISION,
  temp_min_c     DOUBLE PRECISION,
  precip_mm      DOUBLE PRECISION,
  sunshine_h     DOUBLE PRECISION,
  wind_max_kmh   DOUBLE PRECISION,
  rh_mean_pct    DOUBLE PRECISION,
  FOREIGN KEY (city_id) REFERENCES city(city_id)
);
""")

In [7]:
run_sql("""
CREATE TABLE IF NOT EXISTS hotel (
  hotel_id     VARCHAR(16) PRIMARY KEY,
  city_id      VARCHAR(8) NOT NULL,
  hotel_name   TEXT,
  user_rating  DOUBLE PRECISION,
  hotel_url    TEXT,
  description  TEXT,
  lat          DOUBLE PRECISION,
  lon          DOUBLE PRECISION,
  FOREIGN KEY (city_id) REFERENCES city(city_id)
);
""")

In [4]:
def load_df_to_db(df, table_name, engine):
    """Load a DataFrame into a database table, keeping only matching columns."""
    inspector = inspect(engine)
    db_cols = [col["name"] for col in inspector.get_columns(table_name)]
    df_filtered = df[[c for c in df.columns if c in db_cols]]
    df_filtered.to_sql(table_name, engine, if_exists="append", index=False)
    print(f"Loaded {len(df_filtered)} rows into {table_name}")

In [13]:
load_df_to_db(df_coords, "city", engine)
load_df_to_db(df_weather_daily_7d, "weather", engine)
load_df_to_db(df_hotels, "hotel", engine)

Loaded 35 rows into city
Loaded 245 rows into weather
Loaded 100 rows into hotel


In [14]:
with engine.begin() as conn:
    for q in [
        "SELECT COUNT(*) FROM city;",
        "SELECT COUNT(*) FROM weather;",
        "SELECT COUNT(*) FROM hotel;"
    ]:
        n = conn.execute(text(q)).scalar()
        print(q, n)

SELECT COUNT(*) FROM city; 35
SELECT COUNT(*) FROM weather; 245
SELECT COUNT(*) FROM hotel; 100


In [23]:
query = """
SELECT DISTINCT ON (c.city)
  c.city, w.date, w.temp_max_c, w.temp_min_c
FROM city c
JOIN weather w ON c.city_id = w.city_id
ORDER BY c.city, w.date DESC;
"""
df_sample = pd.read_sql(query, engine)
df_sample

,city,date,temp_max_c,temp_min_c
0,Aigues Mortes,2025-11-03,20.8,11.4
1,Aix en Provence,2025-11-03,20.0,11.3
2,Amiens,2025-11-03,14.1,6.7
3,Annecy,2025-11-03,14.7,4.5
4,Ariege,2025-11-03,14.5,3.4
5,Avignon,2025-11-03,19.1,11.6
6,Bayeux,2025-11-03,15.8,11.6
7,Bayonne,2025-11-03,18.2,13.9
8,Besancon,2025-11-03,14.7,7.6
9,Biarritz,2025-11-03,17.7,13.5
